# Start Solr


01 Start Solr
02 Create two config sets - one for hnsw and one for cuvs
03 Create a collection
04 Download the dataset
05 Download the query set
06 Prepare ground truth
07 Create javabin batches using the dataset
## Benchmark using cuVS
08 Upload the batches to Solr and record the indexing time
09 Run the queries and record query time
10 Compute the recall accuracy using the collected search results
## Benchmark using HNSW
11 Delete previous collection and restart Solr
12 Create a new collection
13 Upload the batches to Solr and record the indexing time
14 Run the queries and record query time
15 Compute the recall accuracy using the collected search results
16 Prepare benchmark report

In [ ]:
%%bash
# Start Solr Instance
solr-10.0.0-SNAPSHOT/bin/solr stop -p 8983
export LD_LIBRARY_PATH=/cuvs/cpp/build:$LD_LIBRARY_PATH
solr-10.0.0-SNAPSHOT/bin/solr start -m 12G --force

# Check Solr Status
solr-10.0.0-SNAPSHOT/bin/solr status

In [ ]:
%%bash
cd /workingarea
(cd conf && zip -r - *) | curl -X POST --header "Content-Type:application/octet-stream" --data-binary @- "http://localhost:8983/solr/admin/configs?action=UPLOAD&name=cuvs"
curl "http://localhost:8983/solr/admin/collections?action=CREATE&name=test&numShards=1&collection.configName=cuvs"

In [ ]:
%%bash
# Download dataset, create batches and upload to Solr
cd /workingarea
wget -c -nv https://accounts.searchscale.com/datasets/wikipedia/wiki_dump_5Mx2048D.csv.gz


In [ ]:
%%bash

mkdir batches
java -cp solr-cuvs-benchmarks-1.0-SNAPSHOT-jar-with-dependencies.jar com.searchscale.benchmarks.Indexer data_file=wiki_dump_5Mx2048D.csv.gz output_file=batches/wiki batch_size=50000 docs_count=200000 legacy=true


In [ ]:
%%bash

chmod +x upload_all.sh
./upload_all.sh